In [1]:

!pip -q install faster-whisper gradio librosa soundfile pandas matplotlib jiwer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.7 MB/s eta 0:00:00


In [2]:

import os
import re
import tempfile
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import torch
import gradio as gr

from faster_whisper import WhisperModel

print("Libraries imported successfully!")
print("GPU available:", torch.cuda.is_available())


Libraries imported successfully!
GPU available: False


In [3]:

device = "cuda" if torch.cuda.is_available() else "cpu"

compute_type = "float16" if device == "cuda" else "int8"

model = WhisperModel(
    "tiny.en",
    device=device,
    compute_type=compute_type
)

print("Whisper model loaded successfully!")



Whisper model loaded successfully!


In [4]:

def preprocess_audio(file_path):
    audio, sr = librosa.load(
        file_path,
        sr=16000,
        mono=True
    )

    if len(audio) == 0:
        raise ValueError("The audio file is empty.")

    duration = len(audio) / sr

    if duration < 1:
        raise ValueError("Record at least one second of audio.")

    return audio.astype(np.float32), sr, duration


def transcribe_audio(audio):
    segments, info = model.transcribe(
        audio,
        language="en",
        beam_size=5,
        word_timestamps=True,
        vad_filter=True,
        condition_on_previous_text=False
    )

    segments = list(segments)

    transcript = " ".join(
        segment.text.strip()
        for segment in segments
    ).strip()

    word_data = []

    for segment in segments:
        for word in (segment.words or []):
            word_data.append({
                "word": word.word.strip(),
                "start": float(word.start),
                "end": float(word.end)
            })

    return transcript, word_data


In [5]:

FILLER_WORDS = {"um", "uh", "erm", "hmm", "ah", "like"}

def clean_word(word):
    return re.sub(r"[^a-z']", "", word.lower())


def extract_features(transcript, word_data, duration):
    words = re.findall(
        r"\b[a-zA-Z]+(?:'[a-zA-Z]+)?\b",
        transcript
    )

    total_words = len(words)

    filler_words = [
        clean_word(item["word"])
        for item in word_data
        if clean_word(item["word"]) in FILLER_WORDS
    ]

    filler_count = len(filler_words)

    # Estimated word-gap pauses, not acoustic silence detection
    pause_threshold = 0.75
    pauses = []

    ordered = sorted(word_data, key=lambda x: x["start"])

    for previous, current in zip(ordered, ordered[1:]):
        gap = max(0.0, current["start"] - previous["end"])

        if gap >= pause_threshold:
            pauses.append(round(gap, 2))

    # Pace based on the complete recording
    wpm = (total_words / duration) * 60 if duration else 0

    filler_rate = (
        (filler_count / total_words) * 100
        if total_words else 0
    )

    return {
        "total_words": total_words,
        "duration": round(duration, 2),
        "wpm": round(wpm, 1),
        "filler_count": filler_count,
        "filler_words": filler_words,
        "filler_rate": round(filler_rate, 1),
        "pause_count": len(pauses),
        "pauses": pauses
    }


In [6]:

def evaluate_fluency(features):
    wpm = features["wpm"]
    filler_rate = features["filler_rate"]
    pauses = features["pause_count"]
    duration = features["duration"]

    if features["total_words"] < 5:
        return 0, (
            "Not enough recognized speech. "
            "Record a longer sample for meaningful feedback."
        )

    score = 100
    feedback = []

    # Preferred pace is a configurable coaching heuristic
    if wpm < 90:
        score -= 15
        feedback.append(
            "Try speaking a little faster while staying clear."
        )
    elif wpm > 170:
        score -= 15
        feedback.append(
            "Try slowing down for better clarity."
        )
    else:
        feedback.append(
            "Your speaking pace is within the target range."
        )

    # Filler-word penalty
    score -= min(30, filler_rate * 3)

    if filler_rate > 5:
        feedback.append(
            "Reduce filler words such as um and uh."
        )
    else:
        feedback.append(
            "Your filler-word frequency is relatively low."
        )

    # Normalize pauses to a one-minute recording
    pause_rate = pauses / max(duration / 60, 0.01)

    score -= min(20, pause_rate * 2)

    if pause_rate > 5:
        feedback.append(
            "Several long word gaps were detected. "
            "Practice maintaining a comfortable rhythm."
        )
    else:
        feedback.append(
            "Your long-pause frequency is within the target."
        )

    score = round(max(0, min(100, score)))

    return score, "\n".join(feedback)


In [7]:

def analyze_speech(audio_file):
    if audio_file is None:
        return "Please provide an audio recording.", {}, ""

    try:
        audio, sr, duration = preprocess_audio(audio_file)

        transcript, word_data = transcribe_audio(audio)

        features = extract_features(
            transcript,
            word_data,
            duration
        )

        score, feedback = evaluate_fluency(features)

        results = {
            "Fluency score / 100": score,
            "Words per minute": features["wpm"],
            "Total words": features["total_words"],
            "Filler words": features["filler_count"],
            "Filler rate (%)": features["filler_rate"],
            "Long pauses": features["pause_count"],
            "Duration (seconds)": features["duration"]
        }

        return transcript, results, feedback

    except Exception as error:
        return f"Analysis failed: {error}", {}, ""


gradio interface


In [ ]:

# Close the previous interface if it exists
try:
    demo.close()
except Exception:
    pass

import gradio as gr

with gr.Blocks(title="SpeakSense AI") as demo:

    gr.Markdown("""
    # 🎙️ SpeakSense AI
    ### Intelligent Speech Fluency & Communication Analyzer

    Record your voice or upload an audio file.
    Get your transcript, fluency score and feedback.
    """)

    with gr.Tabs():

        with gr.Tab("🎤 Record Voice"):
            mic_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                format="wav",
                label="Record Your Speech"
            )

            mic_button = gr.Button(
                "Analyze Recording",
                variant="primary"
            )

        with gr.Tab("📁 Upload Audio"):
            upload_input = gr.Audio(
                sources=["upload"],
                type="filepath",
                label="Upload Audio File"
            )

            upload_button = gr.Button(
                "Analyze Uploaded Audio",
                variant="primary"
            )

    transcript_output = gr.Textbox(
        label="Speech Transcript",
        lines=5
    )

    metrics_output = gr.JSON(
        label="Speech Analytics"
    )

    feedback_output = gr.Textbox(
        label="Personalized Feedback",
        lines=5
    )

    outputs = [
        transcript_output,
        metrics_output,
        feedback_output
    ]

    mic_button.click(
        fn=analyze_speech,
        inputs=mic_input,
        outputs=outputs
    )

    upload_button.click(
        fn=analyze_speech,
        inputs=upload_input,
        outputs=outputs
    )

demo.launch(
    share=True,
    debug=True,
    inline=False
)


Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2d1a2393d76408ba68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
